Student Names & Numbers:
Henry Sampers: 23363819
Fuwhad Akinwale: 23362227
David Bradshaw: 23377569

Does the code run: TBD
Link to third part source: 

In [ ]:
#Imports
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

#setting dir paths
data_dir = Path("data/state-farm")
csv_path = data_dir / "driver_imgs_list.csv"
train_dir = data_dir / "imgs" / "train"

#Checks to ensure data is there
print("CSV exists:", csv_path.exists())
print("Training images exist:", train_dir.exists())

#printing classes and counts
df = pd.read_csv(csv_path)
display(df.head())
print(df["classname"].value_counts().sort_index())

In [ ]:
#select 6 random examples from the dataset they will always be same due to the random_state parameter
examples = df.sample(6, random_state=42)

#pairs each image with its data in the driver_imgs_list.csv file and displays it in a 2x3 grid
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, (_, row) in zip(axes.flat, examples.iterrows()):
    image_path = train_dir / row["classname"] / row["img"]
    with Image.open(image_path) as image:
        ax.imshow(image)

    ax.set_title(f'{row["classname"]} | driver {row["subject"]}')
    ax.axis("off")
#actually displays the images
plt.tight_layout()
plt.show()

In [ ]:
#import to split the dataset into training, validation, and test sets while ensuring that images from the same driver are not split across different sets
from sklearn.model_selection import GroupShuffleSplit

# 1 = phone use; 0 = another activity
phone_classes = {"c1", "c2", "c3", "c4"}
df["phone_use"] = df["classname"].isin(phone_classes).astype(int)

print("Binary label counts:")
print(df["phone_use"].value_counts().sort_index())

#Splitting the dataset into training test and validation sets from the training set from kaggle as it only one with labels

# Set aside five complete drivers for the final test set.
test_split = GroupShuffleSplit(n_splits=1, test_size=5, random_state=42)
remaining_idx, test_idx = next(
    test_split.split(df, groups=df["subject"])
)

remaining_df = df.iloc[remaining_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

# Set aside four of the remaining drivers for validation.
val_split = GroupShuffleSplit(n_splits=1, test_size=4, random_state=43)
train_idx, val_idx = next(
    val_split.split(remaining_df, groups=remaining_df["subject"])
)

train_df = remaining_df.iloc[train_idx].reset_index(drop=True)
val_df = remaining_df.iloc[val_idx].reset_index(drop=True)

# Check the sizes, label balance, and driver IDs.
for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(
        name,
        "| images:", len(part),
        "| phone use:", f'{part["phone_use"].mean():.1%}',
        "| drivers:", sorted(part["subject"].unique()),
    )
    assert part["phone_use"].nunique() == 2, f"{name} has only one label"

train_drivers = set(train_df["subject"])
val_drivers = set(val_df["subject"])
test_drivers = set(test_df["subject"])

assert train_drivers.isdisjoint(val_drivers)
assert train_drivers.isdisjoint(test_drivers)
assert val_drivers.isdisjoint(test_drivers)
print("No drivers are shared between sets.")


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class DriverPhoneDataset(Dataset):
    #Match each CSV row to an image and its binary phone use label

    def __init__(self, frame, image_root, transform):
        self.frame = frame.reset_index(drop=True)
        self.image_root = image_root
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]

        #Kaggle stores each image inside the folder for its original class
        image_path = self.image_root / row["classname"] / row["img"]

        with Image.open(image_path) as image:
            image = image.convert("RGB")  #Ensure every image has three channels
            image = self.transform(image)

        #1 means phone use 
        #0 means another activity
        label = int(row["phone_use"])
        return image, label

#Match the papers image size and normalization for the baseline experiment
preprocess = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_dataset = DriverPhoneDataset(train_df, train_dir, preprocess)
val_dataset = DriverPhoneDataset(val_df, train_dir, preprocess)

#Shuffle training batches keep validation order fixed for repeatable evaluation
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False, num_workers=0
)

#Check one batch before connecting the loader to a model
images, labels = next(iter(train_loader))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels in batch:", labels.tolist())

In [ ]:
import torch

#Select examples of both binary labels from the training set
phone_indices = train_df.index[train_df["phone_use"] == 1][:3].tolist()
other_indices = train_df.index[train_df["phone_use"] == 0][:3].tolist()
selected_indices = phone_indices + other_indices

#The tensors were normalized for training; reverse that only for display
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, index in zip(axes.flat, selected_indices):
    image, label = train_dataset[index]
    image = (image * std + mean).clamp(0, 1)

    #Matplotlib expects height, width, channels. PyTorch uses channels first
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title("Phone use" if label == 1 else "Other activity")
    ax.axis("off")

plt.tight_layout()
plt.show()